In [2]:
import fastf1
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from xgboost import XGBClassifier, XGBRegressor, plot_importance
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import shap
import matplotlib.pyplot as plt

/Users/dimkostir/Desktop/Projects/f1-predictions/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Get FP2 laps to train a pre-race model 

In [10]:
master = pd.read_parquet('/Users/dimkostir/Desktop/Projects/f1-predictions/data/processed/master_tables/master_table_24_25.parquet')

In [11]:
def median_pace(year,r_laps, gp):
  pace = r_laps.groupby('Driver')['LapTime'].median()
  pace = pd.DataFrame(pace)
  #pace.to_parquet(f'/Users/dimkostir/Desktop/Projects/F1/data/processed/median_pace/median_pace_{gp}_{year}.parquet')
  pace = pace.reset_index()
  return pace


#years = [2022,2023,2024,2025,2026]

years = [2024,2025]

locations_list = ["Melbourne", "Shanghai", "Suzuka", "Sakhir", "Jeddah",
             "Miami", "Imola", "Monaco", "Barcelona", "Montreal", "Spielberg",
             "Silverstone", "Spa", "Budapest", "Zandvoort", "Monza", "Baku",
             "Singapore", "Austin", "Mexico City", "Sao Paolo", "Las Vegas", "Lusail", "Yas Marina"]

def get_fp2_laps(year,gp):
 fp2 = fastf1.get_session(year, gp, 'FP2')
 fp2.load()
 fp2 = fp2.laps

 fp2 = fp2[
     (fp2["TrackStatus"] == "1") &
     (fp2["LapTime"].notna()) &
     (fp2["PitInTime"].isna()) &
     (fp2["PitOutTime"].isna())].copy()

# Timedelta conversions
 for col in ['Time', 'LapTime', 'LapStartTime',
                'Sector1Time', 'Sector2Time', 'Sector3Time',
                'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime']:
   fp2[col] = fp2[col].dt.total_seconds()
 fp2 = fp2.drop(["PitInTime", "PitOutTime", "LapStartDate"], axis= 1)

 #q_laps.to_parquet(f'/Users/dimkostir/Desktop/Projects/F1/data/processed/q_laps/q_laps_{year}_{gp}.parquet')
 return fp2


def team_dif(year, fp2, gp):
    team_df = fp2[['Driver', 'Team']].drop_duplicates()
    pace = median_pace(year, fp2, gp)
    
    teams_dif = pd.merge(team_df, pace, on='Driver', how='inner')
    teams_dif = pd.merge(teams_dif, teams_dif, on='Team', how='inner')

    teams_dif = teams_dif[teams_dif["Driver_x"] != teams_dif["Driver_y"]]
    teams_dif["team_dif"] = teams_dif["LapTime_x"] - teams_dif["LapTime_y"]
    teams_dif = teams_dif.drop(["Driver_y", "LapTime_y", "LapTime_x"], axis=1)
    teams_dif = teams_dif.rename(columns={"Driver_x": "Driver"})

    #teams_dif.to_parquet(f'/Users/dimkostir/Desktop/Projects/F1/data/processed/teams_dif/teams_dif_{gp}_{year}.parquet')

    return teams_dif

def calc_deg_rate(stint_df):
    if len(stint_df) < 4:
        return None
    slope = np.polyfit(stint_df["TyreLife"], stint_df["LapTime"], 1)[0]
    return slope


def deg_rate(fp2_laps, year, gp):
    fp2_stint2 = fp2_laps
    degradation_rate = fp2_stint2.groupby(["Driver", "Stint"]).apply(calc_deg_rate, include_groups=False)
    degradation_rate = pd.DataFrame(degradation_rate).reset_index()
    degradation_rate = degradation_rate.rename(columns={0: "Deg_Rate"})
    #degradation_rate.to_parquet(f'/Users/dimkostir/Desktop/Projects/F1/data/processed/deg_rate/deg_rate_{year}_{gp}.parquet')
    return degradation_rate

def d_rate_final(fp2, year, gp):
    stint = fp2.groupby(["Driver", "Stint"]).size().reset_index()
    stint = stint.rename(columns={0: "Laps"})
    deg = deg_rate(fp2, year, gp)
    stint_df = pd.merge(stint, deg, on=['Driver', 'Stint'], how='inner')
    merged_df_clean = stint_df[stint_df["Deg_Rate"].notna()]
    result = merged_df_clean.groupby("Driver").apply(
        lambda x: (x["Deg_Rate"] * x["Laps"]).sum() / x["Laps"].sum(), include_groups=False
    ).reset_index()
    result = result.rename(columns={0: "Deg_Rate_Weighted"})
    return result

def circuit_location(year,gp):
   location = fastf1.get_event(year, gp)["Location"]
   return location

In [12]:
def get_fp2_features(year,gp):
    fp2_laps = get_fp2_laps(year, gp)
    fp2_pace = median_pace(year, fp2_laps, gp)
    fp2_dif = team_dif(year, fp2_laps,gp)
    fp2_deg =  d_rate_final(fp2_laps, year, gp)

    fp2_features = pd.merge(fp2_pace, fp2_dif, on='Driver', how='outer')
    fp2_features = pd.merge(fp2_features, fp2_deg, on='Driver', how='outer')
    
    fp2_features = fp2_features.drop('Team', axis=1) 
    fp2_features["Year"] = year
    fp2_features["Location"] = circuit_location(year, gp)

    fp2_features = fp2_features.rename(columns={
        'LapTime': 'fp2_median_pace',
        'team_dif': 'fp2_team_dif',
        'Deg_Rate_Weighted': 'fp2_deg_rate'
    })
    return fp2_features


In [ ]:
import time

all_races = []
success_count = 0
fail_count = 0

for year in [2024, 2025]:  # Μόνο αυτά τα 2 years
    schedule = fastf1.get_event_schedule(year)
    total_rounds = len(schedule[schedule['EventFormat'] != 'testing'])
    
    for race in range(1, total_rounds + 1):
        try:
            master_table2 = get_fp2_features(year, race)
            all_races.append(master_table2)
            success_count += 1
            print(f"✓ {year} round {race} — Success ({success_count} total)")
            time.sleep(8)  # 8 seconds delay μεταξύ races
            
        except Exception as e:
            error_type = type(e).__name__
            fail_count += 1
            
            if "RateLimitExceededError" in error_type:
                print(f"\n⏰ Rate limit at {year} round {race}")
                print("Sleeping 1 hour...")
                time.sleep(3600)
                
                # Retry
                print(f"Retrying {year} round {race}...")
                try:
                    master_table2 = get_fp2_features(year, race)
                    all_races.append(master_table2)
                    success_count += 1
                    print(f"✓ {year} round {race} (retry OK)")
                except Exception as retry_error:
                    print(f"✗ {year} round {race} failed after retry: {retry_error}")
            else:
                print(f"✗ {year} round {race}: {e}")

# Summary
print(f"\n{'='*60}")
print(f"SUMMARY: {success_count} succeeded, {fail_count} failed")
print(f"Total races in all_races: {len(all_races)}")
print(f"{'='*60}\n")

if len(all_races) == 0:
    print("ERROR: No races loaded. Check errors above.")
else:
    # Proceed με merge
    fp2_all = pd.concat(all_races, ignore_index=True)
    print(f"FP2 features shape: {fp2_all.shape}")
    
    master_clean = master.drop(['Median_lap_time', 'team_dif', 'Deg_Rate_Weighted'], axis=1)
    
    master2 = pd.merge(
        master_clean,
        fp2_all,
        on=['Driver', 'Year', 'Location'],
        how='left'
    )
    
    print(f"Master2 shape: {master2.shape}")
    print(f"\nNull counts in FP2 features:")
    print(master2[['fp2_median_pace', 'fp2_team_dif', 'fp2_deg_rate']].isnull().sum())

In [14]:
print(f"FP2 features shape: {fp2_all.shape}")
print(f"Master Phase 2 shape: {master2.shape}")
print(f"\nColumns in master2:\n{master2.columns.tolist()}")
print(f"\nNull counts:\n{master2.isnull().sum()}")

FP2 features shape: (663, 6)
Master Phase 2 shape: (958, 13)

Columns in master2:
['Driver', 'TeamName_x', 'GridPosition', 'Finish_Position', 'Status', 'delta_to_pole', 'Qual_Position', 'Year', 'Location', 'dnf', 'fp2_median_pace', 'fp2_team_dif', 'fp2_deg_rate']

Null counts:
Driver               0
TeamName_x           0
GridPosition         0
Finish_Position      0
Status               0
delta_to_pole       11
Qual_Position        1
Year                 0
Location             0
dnf                  0
fp2_median_pace    298
fp2_team_dif       310
fp2_deg_rate       354
dtype: int64


In [ ]:
master2.drop(columns = ["dnf"], axis= 1)
master2.to_parquet('/Users/dimkostir/Desktop/Projects/f1-predictions/data/processed/master_tables/master_table_phase2_24_25.parquet')
master2.to_csv('/Users/dimkostir/Desktop/Projects/f1-predictions/data/csv/master_table_02.csv')
print("✅ Phase 2 master table saved!")

✅ Phase 2 master table saved!


Final 2022-2026 check

In [4]:
m26 = pd.read_parquet('/Users/dimkostir/Desktop/Projects/f1-predictions/data/processed/final_phase_01/final_phase_01.parquet')
m26.head()

,Driver,TeamName_x,GridPosition,Finish_Position,Status,delta_to_pole,Qual_Position,Year,Location,fp2_median_pace,fp2_team_dif,fp2_deg_rate
0,LEC,Ferrari,1.0,1.0,Finished,0.000,1.0,2022,Sakhir,97.169,-1.4070,NaN
1,SAI,Ferrari,3.0,2.0,Finished,0.129,3.0,2022,Sakhir,98.576,1.4070,1.771703
2,HAM,Mercedes,5.0,3.0,Finished,0.680,5.0,2022,Sakhir,98.827,0.3130,1.508391
3,RUS,Mercedes,9.0,4.0,Finished,1.658,9.0,2022,Sakhir,98.514,-0.3130,1.676536
4,MAG,Haas F1 Team,7.0,5.0,Finished,1.250,7.0,2022,Sakhir,98.620,-1.1895,1.785214
